In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ f = xW + b $$
$$ \frac{\partial f}{\partial x} = W $$
$$ \frac{\partial f}{\partial W} = x $$
$$ \frac{\partial f}{\partial b} = 1 $$

In [ ]:
import torch.autograd as autograd
import torch.nn as nn
import torch as tr

import import_ipynb
from approx import approx # type: ignore


def linear(x: tr.Tensor, W: tr.Tensor, b: tr.Tensor) -> tr.Tensor:
    """Linear function: y = x @ W + b."""

    # (S, out_features) = (S, in_features) @ (in_features, out_features) + (out_features,)
    assert x.shape[1] == W.shape[0]
    return tr.addmm(b, x, W)


class LinearFunction(autograd.Function):
    """
    Custom autograd.Function implementing a linear layer:

        y = x @ W + b

    with explicit forward and backward passes.
    """

    @staticmethod
    def forward(ctx, x: tr.Tensor, W: tr.Tensor, b: tr.Tensor) -> tr.Tensor:
        assert x.shape[1] == W.shape[0]
        ctx.save_for_backward(x, W)
        z = linear(x, W, b)
        return z

    @staticmethod
    def backward(ctx, dF_df: tr.Tensor) -> tuple[tr.Tensor, tr.Tensor, tr.Tensor]:
        (x, W) = ctx.saved_tensors
        (S, FI, FO) = x.shape[0], x.shape[1], W.shape[1]
               
        # (features, output) = (features, samples) @ (samples, output)
        dF_dW = x.t() @ dF_df          
        assert dF_dW.shape == (FI, FO)
        
        # (output,) * (output,) -> (output, )
        dF_db = dF_df.sum(dim=0)     
        assert dF_db.shape == (FO,)
        
        return (None, dF_dW, dF_db)
    
class Linear(nn.Module):
    """Custom module for the affine linear transformation."""

    def __init__(self, in_features: int, out_features: int, W: tr.Tensor = None, b: tr.Tensor = None) -> None:
        super().__init__()

        # `W` has shape (out_features, in_features) to be consistent with `nn.Linear`
        if W is None:
            self.weight = nn.Parameter(tr.randn(out_features, in_features) * 0.01)
        else:
            self.weight = nn.Parameter(W.clone().detach().requires_grad_(True))

        if b is None:
            self.bias = nn.Parameter(tr.randn(out_features, ))
        else:
            self.bias = nn.Parameter(b.clone().detach().requires_grad_(True))

    def forward(self, x):
        assert x.shape[1] == self.weight.shape[1]
        return LinearFunction.apply(x, self.weight.t(), self.bias)
    

In [ ]:
# Unit tests

def test_LinearFunction_gradcheck(S: int, FI: int, FO: int) -> None:
    def fn(x, W, b):
        return LinearFunction.apply(x, W, b)

    x = tr.randn(S, FI, dtype=tr.float64, requires_grad=False)
    W = tr.randn(FI, FO, dtype=tr.float64, requires_grad=True)
    b = tr.randn(FO, dtype=tr.float64, requires_grad=True)

    assert autograd.gradcheck(fn, (x, W, b), eps=1e-6, atol=1e-4, rtol=1e-4)


def test_Linear_compare(S: int, FI: int, FO: int) -> None:
    nn_model = nn.Linear(FI, FO)
    dj_model = Linear(FI, FO, nn_model.weight, nn_model.bias)

    x = tr.randn(S, FI, dtype=tr.float32, requires_grad=True)

    x1 = x.clone().detach().requires_grad_(True)
    y1 = dj_model(x1).sum()
    y1.backward()

    x2 = x.clone().detach().requires_grad_(True)
    y2 = nn_model(x2).sum()
    y2.backward()

    assert x1 == approx(x2)
    assert dj_model.weight == approx(nn_model.weight)
    assert dj_model.weight.grad == approx(nn_model.weight.grad)
    assert dj_model.bias == approx(nn_model.bias)
    assert dj_model.bias.grad == approx(nn_model.bias.grad)


if __name__ == "__main__":
    test_LinearFunction_gradcheck(1, 1, 1)
    test_LinearFunction_gradcheck(10, 1, 1)
    test_LinearFunction_gradcheck(10, 20, 1)
    test_LinearFunction_gradcheck(10, 30, 30)

    test_Linear_compare(1, 1, 1)
    test_Linear_compare(10, 1, 1)
    test_Linear_compare(10, 20, 1)
    test_Linear_compare(10, 30, 30)
